In [22]:
import random
import pandas as pd
import numpy as np
import biovec
import json
import seaborn as sns
import matplotlib.pyplot as plt
from Bio.SeqUtils import MeltingTemp as mt
from Bio.Seq import Seq
from itertools import product
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from biovec.models.duplex_vec import generate_complement
import biovec

In [23]:
random.seed(42)

# Helper Functions

In [24]:
def generate_random_dna_sequence(length: int) -> str:
    """Generate a random DNA sequence of given length."""
    return ''.join(np.random.choice(['A', 'T', 'C', 'G'], size=length))

def sample_dna_sequences(length: int, sample_size: int=4**8) -> list[str]:
    """Randomly sample DNA sequences of a given length with no duplicates."""
    max_seq = 4 ** length
    if sample_size > max_seq:
        raise ValueError(f"Cannot sample {sample_size} unique sequences, exceeds maximum possible sequences {max_seq}")
    
    # Generate unique sequences until sample size is reached.
    seqs = set()
    while len(seqs) < sample_size:
        seq = generate_random_dna_sequence(length)
        seqs.add(seq)
    return list(seqs)

def dna_complement(base: str) -> str:
    """Get the Watson-Crick complement of a DNA base."""
    complement = {
        'A': 'T',
        'T': 'A',
        'C': 'G',
        'G': 'C'
    }
    return complement.get(base, base)  # Return the base itself if not found

def random_mismatch_perturbation(sequence: str, num_mismatches: int=1) -> str:
    """Introduce random mismatches in the sequence."""
    seq_list= list(sequence)
    comp_list = [dna_complement(base) for base in seq_list]
    length = len(seq_list)

    positions = np.random.choice(length, size=num_mismatches, replace=False) # Return numpy array of positions, 'replace' indicates no duplicates

    for i in positions:
        new_base = np.random.choice([b for b in 'ATCG' if b != dna_complement(seq_list[i])])
        comp_list[i] = new_base

    return ''.join(seq_list), ''.join(comp_list)

def generate_dna_sequences(length):
    """Generate a list of all DNA sequences of a given length."""
    return [''.join(seq) for seq in product('ATCG', repeat=length)]

def calculate_duplex_tm(top: str, bottom: str, Na=50, dnac1=50, dnac2=50, saltcorr=5):
    try:
        return mt.Tm_NN(seq=Seq(top), c_seq=Seq(bottom), 
            nn_table=mt.DNA_NN3,   # Allawi & SanaLucia (1997)(default)
            imm_table=mt.DNA_IMM1, # Internal Mismatches: SantaLucia & Peyret, 2001 (default)
            tmm_table=mt.DNA_TMM1, # Terminal Mismatches: (Allawi & SantaLucia, 1997-1998; Peyret et al., 1999; Watkins & SantaLucia, 2005) (default)
            de_table=mt.DNA_DE1,   # Dangling Ends: Bommarito et al. (2000) (default)
            Na=Na, dnac1=dnac1, dnac2=dnac2, saltcorr=saltcorr)
    except:
        return None


def generate_all_ngrams(n: int = 3) -> pd.DataFrame:
    """
    Creates a dataset of all Duplex(Top, Bottom) combinations of length n, 
    preventing missing tokens in downstream XGBoost analysis.

    Returns:
        pd.DataFrame with ["Top", "Bottom", "Mismatches", "Tm"]
    """
    bases = ["A", "C", "G", "T"]
    seqs = ["".join(p) for p in product(bases, repeat=n)]
    rows = []

    for top in tqdm(seqs, desc=f"Generating n={n} Duplexes", leave=True):
        for bottom in seqs:
            tm = calculate_duplex_tm(top, bottom)
            rows.append({
                "Top": top,
                "Bottom": bottom,
                "Mismatches": sum((a != b) for a, b in zip(top, bottom)),
                "Tm": tm
            })
    return pd.DataFrame(rows)

# Revamped Synthethic Dataset
def generate_synthetic_dataset(seq_len: int, mismatch_distribution: dict, n_gram: int) -> pd.DataFrame:
    """
    Generate synthetic dataset with:
      - No mismatches up to a given length, randomly sampled beyond a given threshold
      - A proportional number of mismatches per-length

    Args:
        seq_len (int): Length of sequences (e.g., 10 -> 4^10 top strands)
        mismatch_distribution: Proportion of mismatches to add per-length (e.g. {1: 0.3, 2: 0.2, 3: 0.1})
    Return:
        pd.DataFrame with ["Top", "Bottom", "Mismatches", "Tm"]
    """
        
    rows = []
    
    for length in range(1, seq_len + 1):
        if length <= 10:
            tops = generate_dna_sequences(length) # Enumerates all sequences of this length into a list
        else:
            tops = sample_dna_sequences(length, sample_size=4**8)

        n = len(tops)
        mismatch_count = {mismatches: int(n * frac) for mismatches, frac in mismatch_distribution.items()} # {1: 1000, 2: 300, 3: 700}
        total = n + sum(mismatch_count.values())  # perfect + mismatched

        with tqdm(total=total, desc=f"Generating sequences of length {length}", leave=True) as pbar:
            # Perfect Sequences
            for top in tops:
                bottom = ''.join(dna_complement(b) for b in top)
                tm = calculate_duplex_tm(top, bottom)
                rows.append({"Top": top, "Bottom": bottom, "Mismatches": 0, "Tm": tm})
                pbar.update(1)
            
            # Mismatched Sequences  
            np.random.shuffle(tops)      
            i = 0
            for mismatches, count in mismatch_count.items():
                for _ in range(count):
                    top = tops[i]
                    i += 1
                    top, bottom = random_mismatch_perturbation(top, mismatches)
                    tm = calculate_duplex_tm(top, bottom)
                    rows.append({"Top": top, "Bottom": bottom, "Mismatches": mismatches, "Tm": tm})
                    pbar.update(1)

    # Ensure minimum token coverage for downstream analysis
    df = pd.DataFrame(rows)
    df_coverage = generate_all_ngrams(n_gram)
    df = pd.concat([df, df_coverage]).drop_duplicates(
        subset=["Top", "Bottom"]
    ).reset_index(drop=True)
        
    return df

In [25]:
# Streamline File Management

def mismatch_distribution_to_str(mismatch_dict: dict) -> str:
  """Converts mismatch dictionary or distribution to a string representation."""
  return "_".join(f"m{mismatch}:{frac}" for mismatch, frac in sorted(mismatch_dict.items()))

def create_dataset_id(project_name: str, seq_len: int, mismatch_distribution: str, n_gram: int, seed: int) -> str:
  """Converts a dataset + parameters into a single ID."""
  return f"{project_name}_{seq_len}L_{mismatch_distribution_to_str(mismatch_distribution)}_n{n_gram}_Seed{seed}"

# def create_model_id(n: int, dim: int, window: int, min_count: int, workers: int):
#   """Converts a model + parameters into a single ID."""
#   return f"ModelParams_n{n}_dim{dim}_window{window}_min_count{min_count}_workers{workers}"
# def create_model_id(n: int, size: int, window: int, min_count: int, workers: int, **kwargs) -> str:
#     """
#     Generate a model identifier string based on key hyperparameters.
    
#     Args:
#         n (int): n-gram size
#         size (int): vector dimensionality
#         window (int): context window size
#         min_count (int): minimum word count threshold
#         workers (int): number of worker threads
#         **kwargs: additional parameters (ignored in ID)
    
# #     Returns:
# #         str: a formatted model ID string
# #     """
# #     return f"ModelParams_n{n}_size{size}_window{window}_mincount{min_count}_workers{workers}"
# def create_model_id(**p) -> str:
#     """
#     Build a model ID string from whatever hyperparams are provided.
#     Missing keys are simply skipped.
#     """
#     p = dict(p)  # don't mutate caller's dict
#     if "dim" in p and "size" not in p:  # allow dim alias
#         p["size"] = p.pop("dim")

#     ordered = ("n", "size", "window", "min_count", "workers", "sg", "negative", "hs", "alpha")
#     parts = [f"{k}{p[k]}" for k in ordered if k in p]
#     return "ModelParams_" + "_".join(parts)
# Path Routing
def create_model_id(n: int, size: int, window: int, min_count: int, workers: int, negative: int, alpha: float, sample: float, hs: int) -> str:
  """Converts a model + parameters into a single ID."""
  return f"ModelParams_n{n}_size{size}_window{window}_min_count{min_count}_workers{workers}_negative{negative}_alpha{alpha}_sample{sample}_hs{hs}"
from pathlib import Path
ROOT_FOLDER = Path("Artifacts")

def _create_directory(path: Path) -> Path:
  """Ensure the directory for a path is made and return the path."""
  path.mkdir(parents=True, exist_ok=True)
  return path

def build_dataset_paths(dataset_id: str) -> dict[str, Path]:
  base = _create_directory(ROOT_FOLDER / "Data" / dataset_id)
  return {
    "base": base,
    "raw": base / "raw.parquet",
    "meta": base / "meta.json"
  }

def build_model_paths(dataset_id: str, model_id: str) -> dict[str, Path]:
  base = _create_directory(ROOT_FOLDER / "Model" / dataset_id / model_id)
  return {
    "base": base,
    "model": base / "dv.model",
    "meta": base / "meta.json"
  }

def build_analysis_paths(train_dataset_id: str, model_id: str, encoding_dataset_id= str | None) -> dict[str, Path]:
  root = ROOT_FOLDER / "Analysis" / train_dataset_id / model_id

  if encoding_dataset_id is None or encoding_dataset_id == train_dataset_id:
    base = _create_directory(root)
  else:
    base = _create_directory(root / encoding_dataset_id)

  return {
    "base": base,
    "heatmap": base / "heatmap.png",
    "meta": base / "meta.json"
  }

# Use to write meta-data into JSON
def save_json(path: Path, obj: dict) -> None:
  path.parent.mkdir(parents=True, exist_ok=True)
  with open(path, "w", encoding="utf-8") as f:
    json.dump(obj, f, indent=2, ensure_ascii=False)

# Convenient way to query data and models in these nested folders
def load_dataset(dataset_id: str, which: str="raw") -> pd.DataFrame:
  dataPaths = build_dataset_paths(dataset_id)
  if which not in dataPaths:
    raise ValueError(f"Data not found, select from the following {list(dataPaths.keys())}")
  return pd.read_parquet(dataPaths[which])

def load_model(train_dataset_id: str, model_id: str):
  modelPaths = build_model_paths(train_dataset_id, model_id)
  return biovec.models.load_duplex_model(str(modelPaths["model"]))

In [26]:
def preprocess_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """Drop NaN/negative Tm from DF + return summary statistics"""
    print("Shape Before Processing", df.shape)
    print("Count of Negative Melting Tm:", (df["Tm"] < 0).sum())
    df = df.dropna(subset=["Tm"]).reset_index(drop=True)
    df = df[df["Tm"] >= 0].reset_index(drop=True)

    print("Shape After Processing", df.shape)
    print("")
    print(df["Mismatches"].value_counts().sort_index())
    print("\nSummary Statistics")
    print(df["Tm"].describe())
    print("Dataset successfully processed!")
    return df

# Introduce Buckets
def generate_tm_buckets(df, start, stop, step):
  """
  Assign Tm to a bucket and return bucket labels + counts
  """
  bins = np.arange(start, stop + step, step)
  labels = [f"{bins[i]}–{bins[i+1]}" for i in range(len(bins) - 1)]
  df["Tm Bucket"] = pd.cut(df["Tm"], bins=bins, labels=labels)
  buckets = df["Tm Bucket"].dropna().unique().sort_values().tolist() # [0-10,10-20,30-40]
  return buckets

def bucket_summary(df: pd.DataFrame, buckets):
    print(df["Tm Bucket"].value_counts().reindex(buckets, fill_value=0))

def compute_bucket_similarity(df, vectors, buckets, sample_size=1000, random_state=42):
    """
    Compute approximate mean cosine similarity between Tm buckets
    using subsampling to avoid memory blowup.
    """
    rng = np.random.default_rng(random_state)

    # Normalize vectors once
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    normed = vectors / np.maximum(norms, 1e-12)

    n = len(buckets)
    matrix = np.full((n, n), np.nan)

    for i, b1 in enumerate(buckets):
        idx1 = df.index[df["Tm Bucket"] == b1].to_numpy()

        # Subsample if larger than given sample size
        if len(idx1) > sample_size:
            idx1 = rng.choice(idx1, sample_size, replace=False)

        # Compute within-bucket (diagonal)
        if len(idx1) > 1:  # need at least 2 to compute
            sims = np.dot(normed[idx1], normed[idx1].T)
            # Remove self-similarities (=1.0 on diagonal)
            mean_self = (sims.sum() - len(idx1)) / (len(idx1) * (len(idx1) - 1))
            matrix[i, i] = mean_self


        for j, b2 in enumerate(buckets[i+1:], start=i+1):
            idx2 = df.index[df["Tm Bucket"] == b2].to_numpy()
            if len(idx2) == 0 or len(idx1) == 0:
                continue
            if len(idx2) > sample_size:
                idx2 = rng.choice(idx2, sample_size, replace=False)

            # Compute similarity only on subsamples
            sims = np.dot(normed[idx1], normed[idx2].T).mean()
            matrix[i, j] = sims
            matrix[j, i] = sims

    return matrix

def run_model_analysis(df, dv_model, train_dataset_id: str, model_id: str, analysis_dataset_id: str | None, start: int, stop: int, step: int, sample_size: int = 1000, random_state: int = 42,):
    if analysis_dataset_id is None or analysis_dataset_id == train_dataset_id:
        effective_analysis_id = train_dataset_id
    else:
        effective_analysis_id = analysis_dataset_id
    
    analysisPaths = build_analysis_paths(train_dataset_id, model_id, analysis_dataset_id)
    vectors = dv_model.batch_encode(df)
    buckets = generate_tm_buckets(df, start, stop, step)
    sim_matrix = compute_bucket_similarity(df, vectors, buckets, sample_size, random_state)

    mask = np.tril(np.ones_like(sim_matrix, dtype=bool), k=-1)

    plt.figure(figsize=(8,6))
    sns.heatmap(sim_matrix, xticklabels=buckets, yticklabels=buckets, annot=True, fmt=".2f", mask=mask)
    title = (
        f"Cosine Similarity — {train_dataset_id}"
        if effective_analysis_id == train_dataset_id
        else f"Cosine Similarity — {train_dataset_id} → {effective_analysis_id}"
    )
    plt.title(title)
    plt.tight_layout()
    plt.savefig(analysisPaths["heatmap"])
    plt.close()

    save_json(analysisPaths["meta"], {
        "train_dataset": train_dataset_id,
        "analysis_dataset": effective_analysis_id,
        "model_id": model_id,
        "buckets": buckets,
        "sample_size": sample_size,
        "random_state": random_state,
    })

    return analysisPaths

# Single Simulation Workflow

In [27]:
# FileName Convention: {ProjectName_[SeqLength]L_[MismatchDistribution]_Seed#.parquet}
# Example: PerfectDS_10L_m0:1.0_Seed42.parquet
# Example: Duplex_10L_m0:0.5_m1:0.5_Seed42.parquet

projectName = "Coverage"
seq_len = 10
mismatch_distribution = {1: 0.2, 2: 0.1}
seed = 42
n_gram = 3

df = generate_synthetic_dataset(seq_len=seq_len, mismatch_distribution=mismatch_distribution, n_gram=n_gram)

datasetID = create_dataset_id(projectName, seq_len, mismatch_distribution, n_gram, seed)
dataPaths = build_dataset_paths(datasetID) # returns dictionary of string -> path objects

df.to_parquet(dataPaths["raw"], engine="fastparquet", compression="snappy", index=False)
save_json(dataPaths["meta"], 
          { 
            "project": projectName,
            "seq_len": seq_len,
            "mismatch_distribution": mismatch_distribution,
            "n": n_gram,
            "seed": seed,
            "notes": "Includes n=3 trimer coverage, to avoid missing tokens in Oliveira dataset. Also sequenced up to 4^10 then randomly sample 4**8 afterwards"
            })

print(f'Saved dataset to: {dataPaths["raw"]}')
print(f'Saved dataset meta-data to: {dataPaths["meta"]}')

Generating n=3 Duplexes: 100%|██████████| 64/64 [00:00<00:00, 3707.77it/s]


Saved dataset to: Artifacts/Data/Coverage_10L_m1:0.2_m2:0.1_n3_Seed42/raw.parquet
Saved dataset meta-data to: Artifacts/Data/Coverage_10L_m1:0.2_m2:0.1_n3_Seed42/meta.json


In [28]:
# Query generated dataset, and apply post data-processing
df = load_dataset(datasetID)
print(df["Mismatches"].value_counts().sort_index())
print(df["Tm"].describe().apply(lambda x: format(x, 'f')))

# df = preprocess_dataset(df)

Mismatches
0    1398164
1     280191
2     141523
3       1656
Name: count, dtype: int64
count    1790303.000000
mean          15.369697
std           37.585018
min       -26890.589994
25%           10.771182
50%           19.592210
75%           26.077966
max         3974.450396
Name: Tm, dtype: object


In [29]:
# For self-analysis
buckets = generate_tm_buckets(df, start=0, stop=52, step=4)
bucket_summary(df, buckets)

Tm Bucket
0–4       57385
4–8       77981
8–12     122611
12–16    183600
16–20    250836
20–24    277839
24–28    253913
28–32    178594
32–36     99218
36–40     40292
40–44     11966
44–48      1967
48–52       100
Name: count, dtype: int64


In [30]:
# Train a DuplexVec model from synthethic dataset

# Modify model parameters
n = 3
dim = 64
window = 3
sg = 1 # 1 skip-gram, 0 cbow
min_count = 1
workers = 3
negative = 5
alpha = 0.025
sample = 1e-3

modelID = create_model_id(n=n, size=dim, window=window, min_count=min_count, workers=workers, negative=negative, alpha=alpha, sample=sample, hs=0)
modelPaths = build_model_paths(dataset_id=datasetID, model_id=modelID)

dv = biovec.models.DuplexVec(
  df=df,
  n=n,
  size=dim,
  window=window,
  sg=sg,
  workers=workers,
  corpus_file=str(modelPaths["base"] / "corpus.txt"))

dv.save(str(modelPaths["model"]))
save_json(modelPaths["meta"], {
  "train_dataset": datasetID,
  "params": {
    "n": n, "dim": dim, "window": window, "min_count": min_count, "workers": workers
  }
})
print(f'Saved model to: {modelPaths["model"]}')


Generating corpus file from DataFrame...


Generating corpus: 100%|██████████| 1821534/1821534 [00:20<00:00, 87574.36it/s]


Saved model to: Artifacts/Model/Coverage_10L_m1:0.2_m2:0.1_n3_Seed42/ModelParams_n3_size64_window3_min_count1_workers3_negative5_alpha0.025_sample0.001_hs0/dv.model


In [31]:
dv_model = load_model(datasetID, modelID)

analysisPaths = run_model_analysis(
    df=df,
    dv_model=dv_model,
    train_dataset_id=datasetID,
    model_id=modelID,
    analysis_dataset_id=None,   # self-analysis
    start=0, stop=52, step=4
)

print(f"Analysis saved under {analysisPaths['base']}")

Corpus generation progress: 100%|██████████| 1821534/1821534 [00:19<00:00, 93622.65it/s]


Analysis saved under Artifacts/Analysis/Coverage_10L_m1:0.2_m2:0.1_n3_Seed42/ModelParams_n3_size64_window3_min_count1_workers3_negative5_alpha0.025_sample0.001_hs0


# Train Models with Different Hyperparameters (Single Dataset)

In [32]:
datasetID = "Coverage_10L_m1:0.2_m2:0.1_n3_Seed42"
df = load_dataset(datasetID)
df = preprocess_dataset(df)
# * no Hierarchical Softmax
# * negative sampling: 4 vals [1-20] and 0
# * samples: 4 vals [.4e^-5, 1e^-5)
# * alpha: 4 vals [.021-.04]
# * window sizes (2,5,7)
# * BASELINE: 
#     * vector size: 32
#     * window: 7
#     * n = 3
#     * sg = 1
grid = [
  # Negative sampling sweep
  {"n": 3, "size": 32, "window": 7, "min_count": 1, "workers": 3, "alpha": 0.025, "sample": 1e-3, "negative": 21,  "hs": 1},
  {"n": 3, "size": 32, "window": 7, "min_count": 1, "workers": 3, "alpha": 0.025, "sample": 1e-3, "negative": 28, "hs": 1},
  {"n": 3, "size": 32, "window": 7, "min_count": 1, "workers": 3, "alpha": 0.025, "sample": 1e-3, "negative": 33, "hs": 1},
  {"n": 3, "size": 32, "window": 7, "min_count": 1, "workers": 3, "alpha": 0.025, "sample": 1e-3, "negative": 40, "hs": 1},

  # # Alpha sweep (neg=5, hs=0)
  # {"n": 3, "size": 32,   "window": 7, "min_count": 1, "workers": 3, "alpha": 0.021, "sample": 1e-3, "negative": 5, "hs": 1},
  # {"n": 3, "size": 32,   "window": 7, "min_count": 1, "workers": 3, "alpha": 0.027, "sample": 1e-3, "negative": 5, "hs": 1},
  # {"n": 3, "size": 32,   "window": 7, "min_count": 1, "workers": 3, "alpha": 0.033, "sample": 1e-3, "negative": 5, "hs": 1},
  # {"n": 3, "size": 32,   "window": 7, "min_count": 1, "workers": 3, "alpha": 0.040, "sample": 1e-3, "negative": 5, "hs": 1},

  # # Sample sweep (neg=5, hs=0)
  # {"n": 3, "size": 32,   "window": 7, "min_count": 1, "workers": 3, "alpha": 0.025, "sample": 4e-3,  "negative": 5, "hs": 1},
  # {"n": 3, "size": 32,   "window": 7, "min_count": 1, "workers": 3, "alpha": 0.025, "sample": 5.5e-3,"negative": 5, "hs": 1},
  # {"n": 3, "size": 32,   "window": 7, "min_count": 1, "workers": 3, "alpha": 0.025, "sample": 7e-3,  "negative": 5, "hs": 1},
  # {"n": 3, "size": 32,   "window": 7, "min_count": 1, "workers": 3, "alpha": 0.025, "sample": 8.5e-3,"negative": 5, "hs": 1},

  # # Window sweep (neg=5, hs=0)
  # {"n": 3, "size": 32,   "window": 2, "min_count": 1, "workers": 3, "alpha": 0.025, "sample": 1e-3, "negative": 5, "hs": 1},
  # {"n": 3, "size": 32,   "window": 5, "min_count": 1, "workers": 3, "alpha": 0.025, "sample": 1e-3, "negative": 5, "hs": 1},
  # {"n": 3, "size": 32,   "window": 7, "min_count": 1, "workers": 3, "alpha": 0.025, "sample": 1e-3, "negative": 5, "hs": 1},
]

for params in grid:
  modelID = create_model_id(**params)
  modelPaths = build_model_paths(datasetID, modelID)

  print(f"Training the following model: {modelID}.")

  dv = biovec.models.DuplexVec(
    df=df,
    n=params["n"],
    size=params["size"],
    window=params["window"],
    min_count=params["min_count"],
    workers=params["workers"],
    alpha=params.get("alpha", 0.025),
    negative=params["negative"],
    hs=params["hs"],
    sample=params.get("sample", 1e-3),   # <- float, not list
    corpus_file=str(modelPaths["base"] / "corpus.txt"),
)

  dv.save(str(modelPaths["model"]))
  save_json(modelPaths["meta"], {
    "train_dataset": datasetID,
    "params": params,
  })

  print("Model training complete. Running analysis...")

  run_model_analysis(
        df=df,
        dv_model=dv,
        train_dataset_id=datasetID,
        model_id=modelID,
        analysis_dataset_id=None,  # self-analysis
        start=0, stop=52, step=4
  )

  print(f"Analysis complete for {modelID}")


Shape Before Processing (1821534, 4)
Count of Negative Melting Tm: 233999
Shape After Processing (1556304, 4)

Mismatches
0    1374043
1     164969
2      17291
3          1
Name: count, dtype: int64

Summary Statistics
count    1.556304e+06
mean     2.086300e+01
std      9.814284e+00
min      7.120073e-04
25%      1.497439e+01
50%      2.125021e+01
75%      2.702833e+01
max      3.974450e+03
Name: Tm, dtype: float64
Dataset successfully processed!
Training the following model: ModelParams_n3_size32_window7_min_count1_workers3_negative21_alpha0.025_sample0.001_hs1.
Generating corpus file from DataFrame...


Generating corpus: 100%|██████████| 1556304/1556304 [00:17<00:00, 87366.02it/s]


Model training complete. Running analysis...


Corpus generation progress: 100%|██████████| 1556304/1556304 [00:18<00:00, 83814.43it/s]


Analysis complete for ModelParams_n3_size32_window7_min_count1_workers3_negative21_alpha0.025_sample0.001_hs1
Training the following model: ModelParams_n3_size32_window7_min_count1_workers3_negative28_alpha0.025_sample0.001_hs1.
Generating corpus file from DataFrame...


...

Analysis complete for ModelParams_n3_size32_window7_min_count1_workers3_negative33_alpha0.025_sample0.001_hs1
Training the following model: ModelParams_n3_size32_window7_min_count1_workers3_negative40_alpha0.025_sample0.001_hs1.
Generating corpus file from DataFrame...


Generating corpus: 100%|██████████| 1556304/1556304 [00:18<00:00, 84554.87it/s]


Model training complete. Running analysis...


Corpus generation progress: 100%|██████████| 1556304/1556304 [00:18<00:00, 82545.16it/s]


Analysis complete for ModelParams_n3_size32_window7_min_count1_workers3_negative40_alpha0.025_sample0.001_hs1


# XGBoost Regressor On Experimental Data

In [33]:
from biovec.models.duplex_vec import generate_complement
df_dna = pd.read_csv("dna_melt_temp.csv")
df_dna["Top"] = df_dna["Sequence"].str.upper()
df_dna["Bottom"] = df_dna["Top"].apply(generate_complement)
df_dna = df_dna.rename(columns={"Experimental Temperature": "Tm"})
print(df_dna.shape)
df_dna.head()

(640, 8)


,Sequence,Oligo Concentration (M),Salt Concentration (M),pH,GC Content,Tm,Top,Bottom
0,CCGG,0.0001,1.0,7,1.0000,16.6,CCGG,GGCC
1,CGCG,0.0001,1.0,7,1.0000,23.7,CGCG,GCGC
2,GCGC,0.0001,1.0,7,1.0000,27.5,GCGC,CGCG
3,CCGCGG,0.0001,1.0,7,1.0000,55.2,CCGCGG,GGCGCC
4,CGATCG,0.0001,1.0,7,0.6667,34.3,CGATCG,GCTAGC


In [34]:
df_oliveira = pd.read_csv('Oliveira Table')

LEFT = "CGACGTGC"
RIGHT = "ATGTGCTG"

df_oliveira[["Top", "Bottom"]] = df_oliveira["Centre"].str.split("/", expand=True)
df_oliveira["Top"] = LEFT + df_oliveira["Top"] + RIGHT
df_oliveira["Bottom"] = generate_complement(LEFT) + df_oliveira["Bottom"] + generate_complement(RIGHT)
df_oliveira = df_oliveira.rename(columns={"Temp Exp": "Tm"})
print(df_oliveira.shape)
df_oliveira.head()

(4096, 9)


,Unnamed: 0,i,Centre,Tm,Temp Pred,Melt Index,Mismatches,Top,Bottom
0,0,1,GCG/CGC,69.3,71.47,3.8891,0,CGACGTGCGCGATGTGCTG,GCTGCACGCGCTACACGAC
1,1,2,CGC/GCG,69.1,69.73,3.8559,0,CGACGTGCCGCATGTGCTG,GCTGCACGGCGTACACGAC
2,2,3,GGC/CCG,68.9,69.73,3.8559,0,CGACGTGCGGCATGTGCTG,GCTGCACGCCGTACACGAC
3,3,4,GCC/CGG,68.7,69.73,3.8559,0,CGACGTGCGCCATGTGCTG,GCTGCACGCGGTACACGAC
4,4,5,CGG/GCC,68.2,70.50,3.8707,0,CGACGTGCCGGATGTGCTG,GCTGCACGGCCTACACGAC


In [35]:
df = pd.concat(
  [df_dna[["Top", "Bottom", "Tm"]],
  df_oliveira[["Top", "Bottom", "Tm"]]],
  ignore_index=True
)
print(df.shape)
df.head()

(4736, 3)


,Top,Bottom,Tm
0,CCGG,GGCC,16.6
1,CGCG,GCGC,23.7
2,GCGC,CGCG,27.5
3,CCGCGG,GGCGCC,55.2
4,CGATCG,GCTAGC,34.3


In [36]:
def has_missing_tokens(seq, c_seq, missing, n=3):
    for i in range(len(seq) - n + 1):
        if (seq[i:i+n], c_seq[i:i+n]) in missing:
            return True
    return False

missing = {("AAT", "CAG"), ("AAT", "CAT")} # Need to debug for some reason these tokens aren't in the model already.
mask = df.apply(lambda row: has_missing_tokens(row["Top"], row["Bottom"], missing), axis=1)
df = df[~mask]
df.shape

(4734, 3)

In [37]:
dv = load_model("Coverage_10L_m1:0.2_m2:0.1_n3_Seed42", 
                "ModelParams_n3_dim64_window3_min_count1_workers3")

X = dv.batch_encode(df)
y = df["Tm"].values
print(X.shape)
print(y.shape)
print("Single embedding:\n", X[:1])

Corpus generation progress: 100%|██████████| 4734/4734 [00:00<00:00, 66080.39it/s]

(4734, 64)
(4734,)
Single embedding:
 [[ 0.1172509  -0.17988832  0.15706179  0.17861143 -0.08412637 -0.00832156
   0.14417279 -0.09658194  0.41130614  0.23190752  0.55583674 -0.09850099
  -0.06021911  0.09722096  0.42429245  0.05317045  0.38282746  0.15043782
   0.02889501  0.18616042 -0.03593418  0.4234893  -0.19188109  0.2652374
  -0.10604352  0.34528255 -0.23124398 -0.33351427  0.23133343  0.00072104
  -0.01368362  0.34507182  0.03517445  0.03159529  0.05627104  0.11898285
   0.14608237 -0.00124618  0.08017154 -0.06948774 -0.28529343 -0.06275156
   0.2257703   0.0192472  -0.01349183 -0.03246634  0.11453111  0.21795604
   0.09600918  0.24433573  0.4112296  -0.13429186  0.18513891  0.20751242
   0.28617     0.18988429 -0.0279187   0.1613583  -0.16570225 -0.05306055
   0.33106086 -0.10593447  0.15608442 -0.17097402]]


In [38]:
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error, r2_score, mean_absolute_error

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
xgb.fit(X_train, y_train)

y_pred = xgb.predict(X_test)
print("RMSE:", root_mean_squared_error(y_test, y_pred))
print("R²:", r2_score(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))

RMSE: 3.434901647454864
R²: 0.8082434791897186
MAE: 2.245952957459463


In [39]:
# go thru all sequences, calc mt w/biopython. whats experiemntal errror vs. our model
# go thru test set, all seq in x_test -> run through biopython -> get dataset aggregates RMSE R2 MAE
# make a table cols = model, rows = metrics (RMSE, R2, MAE)

In [ ]:
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score, mean_absolute_error

X_train2, X_test2, y_train2, y_test2, df_train, df_test = train_test_split(
    X, y, df, test_size=0.2, random_state=42
)

# Compute Biopython predictions and coerce failures to NaN
biopython_series = df_test.apply(
    lambda row: calculate_duplex_tm(row["Top"], row["Bottom"], Na=row["Salt Concentration (M)"], dnac1=row["Oligo Concentration (M)"], dnac2=row["Oligo Concentration (M)"], saltcorr=5), axis=1
)
biopython_pred = pd.to_numeric(biopython_series, errors="coerce").to_numpy()

# XGBoost predictions on the test encoding
xgb_pred = xgb.predict(X_test2)

# Build a comparison DataFrame and drop rows with any NaN before metric computation
results_df = pd.DataFrame({"y_true": y_test2, "xgb": xgb_pred, "biopython": biopython_pred})
before = len(results_df)
results_df = results_df.dropna(subset=["y_true", "xgb", "biopython"]).reset_index(drop=True)
dropped = before - len(results_df)
print(f"Dropped {dropped} rows with NaN predictions from comparison")

metrics = {
    "RMSE": lambda y_true, y_pred: np.sqrt(mean_squared_error(y_true, y_pred)),
    "R2": lambda y_true, y_pred: r2_score(y_true, y_pred),
    "MAE": lambda y_true, y_pred: mean_absolute_error(y_true, y_pred),
}

results = pd.DataFrame(
    {
        "XGBoost": [metrics[m](results_df["y_true"], results_df["xgb"]) for m in metrics],
        "Biopython": [metrics[m](results_df["y_true"], results_df["biopython"]) for m in metrics],
    },
    index=list(metrics.keys()),
)

print(results)


Dropped 560 rows with NaN predictions from comparison
       XGBoost  Biopython
RMSE  4.854950  13.443638
R2    0.796354  -0.561494
MAE   3.354727  11.717751
